# DSA4020A – Natural Language Processing

# Semester Project

## Machine Translation of Public Service Announcements (PSAs) in Kenya

**Student:** Levin Ekuam

**Course:** DSA4020A – Natural Language Processing

**Project Duration:** 4 Weeks

---

# Notebook 1: Data Collection and Dataset Curation

## Introduction

Public Service Announcements (PSAs) are short, informative messages issued by governments, non-governmental organizations, and public institutions to educate, warn, or guide the public. In Kenya, PSAs are published in English and Kiswahili across domains such as health, education, agriculture, governance, and public safety.

This notebook focuses on collecting a high-quality dataset of PSAs from reliable Kenyan sources. The dataset created in this notebook will be used for preprocessing, exploratory data analysis, machine translation model training, and evaluation in subsequent notebooks.

---

## Objectives

The objectives of this notebook are:

- Identify reliable PSA sources.
- Collect Public Service Announcements from official institutions.
- Organize the collected PSAs into a structured dataset.
- Store the dataset for further preprocessing and analysis.
- Ensure the collected data is ethically sourced and properly documented.

# Importing Required Libraries

Several Python libraries are required for this stage of the project.

- **pandas** is used to organize collected data into structured tables.
- **requests** downloads webpage content.
- **BeautifulSoup** extracts useful information from HTML pages.
- **os** manages directories and files.
- **datetime** records the date and time of data collection.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime

print("All libraries imported successfully.")

All libraries imported successfully.


# Creating the Project Directories

To maintain an organized workflow, this notebook creates the required folders for storing raw and processed datasets.

The directories are created only if they do not already exist.

In [ ]:
folders = [
    "../data/raw",
    "../data/processed"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders are ready.")

Project folders are ready.


# Defining the Dataset Structure

To ensure consistency throughout the project, each collected PSA will contain the following attributes:

| Column | Description |
|---------|-------------|
| PSA_ID | Unique identifier for each Public Service Announcement |
| Domain | Category of the PSA (Health, Education, Agriculture, Security, Governance) |
| English | English version of the PSA |
| Kiswahili | Kiswahili translation (if available) |
| Source | Organization that published the PSA |
| Date | Publication or collection date |
| URL | Original source URL |

In [ ]:
columns = [
    "PSA_ID",
    "Domain",
    "English",
    "Kiswahili",
    "Source",
    "Date",
    "URL"
]

psa_df = pd.DataFrame(columns=columns)

psa_df.head()

,PSA_ID,Domain,English,Kiswahili,Source,Date,URL


# Project Configuration

Instead of placing all functionality inside the notebook, reusable Python modules are stored in the `SRC` directory.

This modular design improves readability, code reuse, testing, and maintainability. The notebook focuses on explaining the workflow, while reusable functions are implemented in standalone Python files.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from SRC.config import *
from SRC.utils import *
from SRC.scraper import *

# Data Collection Strategy

The quality of a machine translation model depends heavily on the quality of the training data. Rather than collecting text from random websites, this project uses official government and non-governmental organizations that publish Public Service Announcements (PSAs).

The selected sources satisfy three important requirements:

- They are authoritative and reliable.
- They regularly publish public advisories.
- They contain information relevant to Kenyan citizens.

## Target Domains

The project collects PSAs from five domains:

| Domain | Example Sources |
|---------|-----------------|
| Health | Ministry of Health, Kenya Red Cross |
| Education | Ministry of Education |
| Agriculture | Ministry of Agriculture |
| Security | National Police Service, NTSA |
| Governance | IEBC, eCitizen |

Using multiple sources increases the diversity of the dataset and improves the robustness of the translation model.

# Defining Data Sources

To keep track of where each Public Service Announcement originates, we create a table of trusted sources.

This information will later be used for metadata, reproducibility, and citation.

In [ ]:
sources = [
    {
        "Source": "Ministry of Health",
        "Domain": "Health",
        "Website": "https://www.health.go.ke"
    },
    {
        "Source": "Kenya Red Cross",
        "Domain": "Health",
        "Website": "https://www.redcross.or.ke"
    },
    {
        "Source": "Ministry of Education",
        "Domain": "Education",
        "Website": "https://www.education.go.ke"
    },
    {
        "Source": "Ministry of Agriculture",
        "Domain": "Agriculture",
        "Website": "https://www.kilimo.go.ke"
    },
    {
        "Source": "National Police Service",
        "Domain": "Security",
        "Website": "https://www.nationalpolice.go.ke"
    },
    {
        "Source": "IEBC",
        "Domain": "Governance",
        "Website": "https://www.iebc.or.ke"
    },
    {
        "Source": "eCitizen",
        "Domain": "Governance",
        "Website": "https://www.ecitizen.go.ke"
    }
]

sources_df = pd.DataFrame(sources)

sources_df

,Source,Domain,Website
0,Ministry of Health,Health,https://www.health.go.ke
1,Kenya Red Cross,Health,https://www.redcross.or.ke
2,Ministry of Education,Education,https://www.education.go.ke
3,Ministry of Agriculture,Agriculture,https://www.kilimo.go.ke
4,National Police Service,Security,https://www.nationalpolice.go.ke
5,IEBC,Governance,https://www.iebc.or.ke
6,eCitizen,Governance,https://www.ecitizen.go.ke


# Inspecting a Website Before Scraping

Before writing a web scraper, it is important to inspect the HTML structure of the target website.

This allows us to identify:

- article titles
- announcement text
- publication dates
- links

Writing a scraper only after understanding the page structure makes the code more reliable and easier to maintain.

In [ ]:
from SRC.scraper import inspect_page

In [ ]:
redcross_url = "https://www.redcross.or.ke"

inspect_page(redcross_url)

PAGE TITLE
Kenya Red Cross – Always There

FIRST 10 HEADINGS
- Drought Crisisin Kenya
- Become aKenya Red Cross Volunteer
- Floodsin Kenya
- Become aKenya Red Cross Volunteer
- Drought Crisisin Kenya
- Floodsin Kenya
- OurFundamentalPrinciples
- WhatWeDo
- Disaster Management
- Health


<!DOCTYPE html>

<html lang="en-US">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1, maximum-scale=1" name="viewport"/>
<link href="//gmpg.org/xfn/11" rel="profile"/>
<title>Kenya Red Cross – Always There</title>
<meta content="max-image-preview:large" name="robots"/>
<script id="cookieyes" src="https://cdn-cookieyes.com/client_data/ee1fc5f19b9a62b80134e52a/script.js" type="text/javascript"></script><link href="//fonts.googleapis.com" rel="dns-prefetch"/>
<link href="https://redcross.or.ke/feed/" rel="alternate" title="Kenya Red Cross » Feed" type="application/rss+xml"/>
<link href="https://redcross.or.ke/comments/feed/" rel="alternate" title="Kenya Red Cross » Comments Feed" type="application/rss+xml"/>
<link href="https://redcross.or.ke/wp-json/oembed/1.0/embed?url=https%3A%2F%2Fredcross.or.ke%2F" rel="alternate" title="oEmbed (JSON)" type="application/json+oembed"/>
<link href="https://redcross.or.ke/wp-json/oembed/1.0/embed?url=https%3A%2F%2Fredcr

In [ ]:
from SRC.collectors import collect_redcross

articles = collect_redcross()

len(articles)

pd.DataFrame(articles).head(10)